# 02: Cleaning and merging

**Owner:** John (Data Engineering & Acquisition Analyst) · **Ticket:** #4

Runs the team's cleaning pipeline (`capstone.clean`, via `uv run build-dataset`) and records what each step changed. This is the evidence behind the "Data Cleaning and Preparation" report section (`report/data-cleaning.md`).

The cleaning rules come from the data-quality notes in `docs/data-dictionary.md`. The notebook doesn't clean anything itself; it measures what the pipeline did.

In [1]:
import pandas as pd

from capstone.clean import clean_animes, clean_reviews, parse_aired
from capstone.data import RAW_DIR
from capstone.dataset import ANIME_PATH, REVIEWS_PATH, build_dataset

build_dataset()
anime = pd.read_parquet(ANIME_PATH)
reviews = pd.read_parquet(REVIEWS_PATH)

raw_animes = pd.read_csv(RAW_DIR / "animes.csv")
raw_reviews = pd.read_csv(RAW_DIR / "reviews.csv")
raw_profiles = pd.read_csv(RAW_DIR / "profiles.csv")

## 1. Duplicates removed

Before and after row counts for each step.

In [2]:
def step_counts(raw, key):
    exact = raw.drop_duplicates()
    return {"raw rows": len(raw), "after exact duplicates": len(exact), "one row per key": exact[key].nunique()}

pd.DataFrame({
    "animes": step_counts(raw_animes, "uid"),
    "reviews": step_counts(raw_reviews, "uid"),
    "profiles": step_counts(raw_profiles, "profile"),
}).T

,raw rows,after exact duplicates,one row per key
animes,19311,16368,16216
reviews,192112,130519,130519
profiles,81727,47902,47885


In [3]:
# Conflicting copies (same key, different values) and how each was resolved
exact = raw_animes.drop_duplicates()
conflicted = exact[exact.duplicated("uid", keep=False)]
kept_max = (anime.set_index("uid")["members"] == raw_animes.groupby("uid")["members"].max()).all()
print(f"anime with conflicting copies: {conflicted['uid'].nunique()} -> kept the copy with the most members: {kept_max}")

exact_p = raw_profiles.drop_duplicates()
print(f"profiles with conflicting copies: {exact_p[exact_p.duplicated('profile', keep=False)]['profile'].nunique()} -> kept the longest favorites list")

anime with conflicting copies: 152 -> kept the copy with the most members: True
profiles with conflicting copies: 17 -> kept the longest favorites list


## 2. Invalid values removed or blanked

In [4]:
unique_reviews = raw_reviews.drop_duplicates(subset="uid")
invalid = unique_reviews.loc[~unique_reviews["score"].between(1, 10), ["uid", "anime_uid", "score"]]
print(f"reviews dropped for a score outside 1-10: {len(invalid)}")
invalid

reviews dropped for a score outside 1-10: 2


,uid,anime_uid,score
63039,6424,226,0
119190,252847,413,11


In [5]:
sub_cols = ["story_score", "animation_score", "sound_score", "character_score", "enjoyment_score"]
unrated = reviews[sub_cols].isna()
print(f"reviews with at least one unrated sub-score (raw value 0, now blank): {unrated.any(axis=1).sum():,} ({unrated.any(axis=1).mean():.1%})")
(unrated.mean() * 100).round(2).rename("unrated_%").to_frame()

reviews with at least one unrated sub-score (raw value 0, now blank): 7,495 (5.7%)


,unrated_%
story_score,4.53
animation_score,4.58
sound_score,4.81
character_score,4.64
enjoyment_score,4.86


In [6]:
# Airing ranges whose end date fell before the start date (MyAnimeList data-entry errors): end date blanked
unique_animes = raw_animes.drop_duplicates(subset="uid")
inverted = unique_animes["aired"].map(
    lambda t: " to " in t and not t.endswith("?") and parse_aired(t).end is None
)
print(f"anime with an end date before the start date: {inverted.sum()}")
unique_animes.loc[inverted, ["uid", "title", "aired"]]

anime with an end date before the start date: 6


,uid,title,aired
1154,33126,Kurogane no Linebarrels Picture Drama,"Dec 14, 2009 to Aug 26, 2009"
2694,3705,Flashback Game,"Oct 18, 2001 to Apr 18, 2001"
3019,3584,Injuu Nerawareta Idol,"Jul 2, 1997 to Jun 2, 1997"
8939,11829,Manga Mitokoumon,"Sep 3, 1981 to Jul 15, 1981"
12280,34234,Ohayou! Kokekkou-san,"Oct 2, 2016 to Sep 17, 2016"
14177,38571,Tobidase! Dokan-kun,"Oct 26, 2011 to Mar 14, 2011"


## 3. Formats corrected

In [7]:
raw_text = unique_reviews.loc[unique_reviews["score"].between(1, 10), "text"].reset_index(drop=True)
collapsed = raw_text.map(lambda t: " ".join(t.split()))
print(f"reviews that had the 'more pics' + sub-score header: {collapsed.str.startswith('more pics Overall').mean():.1%}")
print(f"cleaned reviews still starting with 'more pics': {reviews['text'].str.startswith('more pics').sum()}")
print(f"median words before / after: {collapsed.str.split().map(len).median():.0f} / {reviews['text'].str.split().map(len).median():.0f}")

reviews that had the 'more pics' + sub-score header: 100.0%
cleaned reviews still starting with 'more pics': 0


median words before / after: 360 / 345


In [8]:
pd.DataFrame({
    "aired -> start_date, end_date, start_year, is_airing": [f"{anime['start_date'].notna().mean():.1%} have a start date; {anime['is_airing'].sum()} still airing at scrape time"],
    "genre -> list": [f"{anime['genre'].map(len).sum():,} genre tags across {len(anime):,} anime"],
    "scores -> five sub-score columns": [f"{len(sub_cols)} integer columns, 1-10 or blank"],
    "episodes -> whole numbers": [f"dtype {anime['episodes'].dtype}"],
}, index=["result"]).T

,result
"aired -> start_date, end_date, start_year, is_airing",98.3% have a start date; 438 still airing at s...
genre -> list,"46,589 genre tags across 16,216 anime"
scores -> five sub-score columns,"5 integer columns, 1-10 or blank"
episodes -> whole numbers,dtype Int64


## 4. Missing values: strategy per column

Nothing is imputed. Imputing a success measure (`score`, `members`) would put invented values into the evidence for an investment decision. Missing values stay blank (NaN), and each analysis filters to the rows it can use.

In [9]:
strategy = {
    "synopsis": "Keep blank. Text only; not used as a numeric feature.",
    "episodes": "Keep blank. Mostly shows still airing; filter out when episodes is a model input.",
    "ranked": "Keep blank. Use `score` as the rating measure instead.",
    "score": "Keep blank. Exclude from rating-based analysis; these titles are not acquisition candidates.",
    "img_url": "Keep blank. Not used in analysis.",
    "start_date": "Keep blank. 'Not available' on MyAnimeList.",
    "start_year": "Keep blank. Same rows as start_date.",
    "end_date": "Keep blank. Still airing, 'Not available', or an end date before the start.",
    "mean_review_score": "Keep blank. Anime with no reviews (n_reviews = 0); filter on n_reviews.",
}
missing = (anime.isna().mean() * 100).round(2)
missing = missing[missing > 0].rename("missing_%").to_frame()
sub_mean_note = "Keep blank. No reviews, or none of the anime's reviews rated this category."
missing["strategy"] = missing.index.map(lambda c: strategy.get(c, sub_mean_note))
missing

,missing_%,strategy
synopsis,4.71,Keep blank. Text only; not used as a numeric f...
episodes,3.03,Keep blank. Mostly shows still airing; filter ...
ranked,10.26,Keep blank. Use `score` as the rating measure ...
score,2.10,Keep blank. Exclude from rating-based analysis...
img_url,1.02,Keep blank. Not used in analysis.
start_date,1.68,Keep blank. 'Not available' on MyAnimeList.
end_date,4.42,"Keep blank. Still airing, 'Not available', or ..."
start_year,1.68,Keep blank. Same rows as start_date.
mean_review_score,49.97,Keep blank. Anime with no reviews (n_reviews =...
mean_story_score,51.15,"Keep blank. No reviews, or none of the anime's..."


## 5. Validation checks

Each check fails loudly if a cleaning rule is broken.

In [10]:
checks = {
    "one row per anime": anime["uid"].is_unique,
    "one row per review": reviews["uid"].is_unique,
    "review scores within 1-10": reviews["score"].between(1, 10).all(),
    "sub-scores within 1-10 or blank": reviews[sub_cols].stack().between(1, 10).all(),
    "no review text keeps the page header": not reviews["text"].str.startswith("more pics").any(),
    "no end date before its start date": not (anime["start_date"] > anime["end_date"]).any(),
    "every review counted once in n_reviews": anime["n_reviews"].sum() == len(reviews),
    "every review belongs to a cleaned anime": reviews["anime_uid"].isin(anime["uid"]).all(),
    "anime with reviews == anime with a mean review score": (anime["n_reviews"] > 0).sum() == anime["mean_review_score"].notna().sum(),
}
for name, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {name}")
assert all(checks.values())

PASS  one row per anime
PASS  one row per review
PASS  review scores within 1-10
PASS  sub-scores within 1-10 or blank
PASS  no review text keeps the page header
PASS  no end date before its start date
PASS  every review counted once in n_reviews
PASS  every review belongs to a cleaned anime
PASS  anime with reviews == anime with a mean review score


## 6. Review coverage caveat

`n_reviews` counts the reviews **in this dataset**, which depends on what the scraper collected. It is not MyAnimeList's total. Two signs that coverage is incomplete for some titles:

- a spike of anime with **exactly 20 reviews**, which looks like a one-page (20-review) scrape limit;
- popular titles with **no reviews at all**.

So `n_reviews` should not be used as a measure of review volume or popularity; use `members` and `n_favorites` instead. Mean review scores remain usable, since 20 reviews is still a reasonable sample.

In [11]:
counts = anime.loc[anime["n_reviews"] > 0, "n_reviews"].value_counts().sort_index()
print("anime by exact review count (note the spike at 20):")
print(counts.loc[15:25].to_string())

anime by exact review count (note the spike at 20):
n_reviews
15     70
16     69
17     54
18     42
19     47
20    172
21     52
22     44
23     37
24     34
25     47


In [12]:
top500 = anime.nsmallest(500, "popularity")
print(f"500 most popular anime: {(top500['n_reviews'] == 20).sum()} have exactly 20 reviews, {(top500['n_reviews'] == 0).sum()} have none")
top500.loc[top500["n_reviews"] == 0, ["title", "popularity", "ranked", "members", "n_favorites"]]

500 most popular anime: 50 have exactly 20 reviews, 6 have none


,title,popularity,ranked,members,n_favorites
6435,Hunter x Hunter (2011),20,3.0,1052761,3149
8971,Yahari Ore no Seishun Love Comedy wa Machigatt...,146,258.0,479961,508
3024,To LOVE-Ru,220,3059.0,395149,73
11562,Rokudenashi Majutsu Koushi to Akashic Records,274,2520.0,347381,27
3213,Zero no Tsukaima: Princesses no Rondo,304,1749.0,319571,16
9871,Grisaia no Rakuen,422,608.0,240650,67


## 7. Final tables

In [13]:
print(f"anime.parquet:   {len(anime):,} rows x {anime.shape[1]} columns  ({ANIME_PATH.stat().st_size / 1e6:.1f} MB)")
print(f"reviews.parquet: {len(reviews):,} rows x {reviews.shape[1]} columns  ({REVIEWS_PATH.stat().st_size / 1e6:.1f} MB)")
anime.dtypes.astype(str).rename("dtype").to_frame()

anime.parquet:   16,216 rows x 23 columns  (5.6 MB)
reviews.parquet: 130,517 rows x 11 columns  (213.8 MB)


,dtype
uid,int64
title,str
synopsis,str
genre,object
episodes,Int64
members,int64
popularity,int64
ranked,float64
score,float64
img_url,str


In [14]:
anime.sort_values("members", ascending=False).head(5)[
    ["title", "members", "score", "n_reviews", "mean_review_score", "n_favorites", "start_year", "genre"]
]

,title,members,score,n_reviews,mean_review_score,n_favorites,start_year,genre
1382,Death Note,1871043,8.65,854,8.809133,3360,2006,"[Mystery, Police, Psychological, Supernatural,..."
7402,Shingeki no Kyojin,1754979,8.47,20,8.250000,1966,2013,"[Action, Military, Mystery, Super Power, Drama..."
6573,Sword Art Online,1657823,7.49,1292,6.390093,1303,2012,"[Action, Adventure, Fantasy, Game, Romance]"
3946,Fullmetal Alchemist: Brotherhood,1615084,9.23,637,9.243328,4914,2009,"[Action, Military, Adventure, Comedy, Drama, M..."
10383,One Punch Man,1475219,8.68,414,8.185990,1222,2015,"[Action, Sci-Fi, Comedy, Parody, Super Power, ..."
